# Fake News Detection: Loss Curves and Evaluation Figures (reproducible)

This notebook rebuilds, from scratch, every diagnostic figure in Chapter 4 of the thesis: the **loss curves (Fig. 4.4)**, the confusion matrices, the ROC curves and the precision-recall curves. Nothing here is hard-coded. Each number and curve is produced by training the models and reading the real predictions.

**What it does**
1. Downloads the two datasets from the Hugging Face Hub: ISOT (headlines) and LIAR2 (statements).
2. Cleans them (removes the "(Reuters)" dateline, URLs and duplicate headlines) and makes a stratified 70:15:15 train, validation and test split (seed 0).
3. Trains three models on each dataset for 3 epochs: **BiLSTM**, **BERT**, **BERT + BiLSTM**.
4. Records the training loss and validation loss after every epoch, restores the weights of the epoch with the lowest validation loss, and predicts the test set.
5. Draws the figures from those recorded values.

**How to run:** set the runtime to a GPU (Runtime > Change runtime type > T4 GPU), then choose Runtime > Run all. A full run takes roughly 30 to 40 minutes on a T4. To try it in about two minutes first, set `QUICK = True` in the settings cell.

Small differences from the thesis numbers (in the third decimal) are normal, because the thesis runs used a different GPU back end.

In [ ]:
!pip -q install transformers huggingface_hub scikit-learn matplotlib

In [ ]:
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from sklearn.metrics import (accuracy_score, auc, confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from torch import nn
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

QUICK = False
SEED, EPOCHS, BATCH, MAX_LEN = 0, 3, 32, 64
BERT = "bert-base-uncased"
OUT = Path("results")
OUT.mkdir(exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEV)

## 1. Data

ISOT label 0 means fake. For LIAR2 the original labels 0, 1 and 2 (pants-on-fire, false, barely-true) are treated as fake, and 3, 4 and 5 as real. In the arrays below `1` always means fake.

In [ ]:
ISOT_REPO = "ErfanMoosaviMonazzah/fake-news-detection-dataset-English"
LIAR2_REPO = "chengxuphd/liar2"
DATELINE = re.compile(r"^[A-Z][A-Za-z .,/'-]{1,40}\s*\((reuters)\)\s*[-\u2013\u2014]?\s*", re.I)


def clean(text):
    text = DATELINE.sub("", str(text))
    text = re.sub(r"\breuters\b", " ", text, flags=re.I)
    text = re.sub(r"https?://\S+|pic\.twitter\.com/\S+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def load(name):
    if name == "isot":
        files = [hf_hub_download(ISOT_REPO, f"{s}.tsv", repo_type="dataset") for s in ("train", "validation", "test")]
        df = pd.concat([pd.read_csv(f, sep="\t") for f in files])
        df = df.dropna(subset=["title"]).drop_duplicates(subset="title")
        out = pd.DataFrame({"text": df["title"], "label": (df["label"] == 0).astype(int)})
    else:
        files = [hf_hub_download(LIAR2_REPO, f"{s}.csv", repo_type="dataset") for s in ("train", "valid", "test")]
        df = pd.concat([pd.read_csv(f) for f in files])
        out = pd.DataFrame({"text": df["statement"], "label": df["label"].isin({0, 1, 2}).astype(int)})
    out["text"] = out["text"].apply(clean)
    return out[out["text"].str.len() > 0].reset_index(drop=True)


def split(df, seed):
    tr, rest = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=seed)
    va, te = train_test_split(rest, test_size=0.5, stratify=rest["label"], random_state=seed)
    return tr, va, te


DATA = {}
for name in ("isot", "liar"):
    df = load(name)
    tr, va, te = split(df, SEED)
    if QUICK:
        tr, va, te = tr.sample(600, random_state=0), va.sample(200, random_state=0), te.sample(300, random_state=0)
    DATA[name] = (tr, va, te)
    print(f"{name}: {len(df)} samples, fake share {df['label'].mean():.3f} | train/val/test = {len(tr)}/{len(va)}/{len(te)}")

## 2. Models

* **BiLSTM:** a 2-layer bidirectional LSTM (hidden size 128, dropout 0.5) on randomly initialised word-piece embeddings, with max-pooling over time.
* **BERT:** `bert-base-uncased` fine-tuned with a classification head.
* **BERT + BiLSTM:** the full sequence of BERT's contextual embeddings goes into a BiLSTM (hidden size 128), then a dense layer with ReLU, dropout 0.3 and the output layer.

All three use cross-entropy loss, AdamW, a linear learning-rate schedule with 6% warm-up, gradient clipping at 1.0, batch size 32, maximum length 64 and 3 epochs.

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab, dim=128, hidden=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim, padding_idx=0)
        self.lstm = nn.LSTM(dim, hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=0.5)
        self.drop = nn.Dropout(0.5)
        self.out = nn.Linear(2 * hidden, 2)

    def forward(self, ids, mask):
        seq, _ = self.lstm(self.emb(ids))
        pooled = seq.masked_fill(~mask.bool().unsqueeze(-1), -1e4).max(dim=1).values
        return self.out(self.drop(pooled))


class BertClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = AutoModelForSequenceClassification.from_pretrained(BERT, num_labels=2)

    def forward(self, ids, mask):
        return self.net(input_ids=ids, attention_mask=mask).logits


class BertBiLSTMClassifier(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.bert = AutoModel.from_pretrained(BERT)
        self.drop_in = nn.Dropout(0.1)
        self.lstm = nn.LSTM(self.bert.config.hidden_size, hidden, batch_first=True, bidirectional=True)
        self.dense = nn.Linear(2 * hidden, hidden)
        self.drop_out = nn.Dropout(0.3)
        self.out = nn.Linear(hidden, 2)

    def forward(self, ids, mask):
        states = self.drop_in(self.bert(input_ids=ids, attention_mask=mask).last_hidden_state)
        seq, _ = self.lstm(states)
        pooled = seq.masked_fill(~mask.bool().unsqueeze(-1), -1e4).max(dim=1).values
        return self.out(self.drop_out(torch.relu(self.dense(pooled))))


SETTINGS = {
    "bilstm": {"lr": 1e-3, "weight_decay": 0.0},
    "bert": {"lr": 2e-5, "weight_decay": 0.01},
    "bert_bilstm": {"lr": 2e-5, "weight_decay": 0.01},
}
BUILD = {"bilstm": lambda tok: BiLSTMClassifier(tok.vocab_size), "bert": lambda tok: BertClassifier(), "bert_bilstm": lambda tok: BertBiLSTMClassifier()}

In [ ]:
def batches(tok, frame, shuffle, rng):
    idx = rng.permutation(len(frame)) if shuffle else np.arange(len(frame))
    texts, labels = frame["text"].tolist(), frame["label"].tolist()
    for i in range(0, len(idx), BATCH):
        j = idx[i:i + BATCH]
        enc = tok([texts[k] for k in j], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
        yield enc["input_ids"].to(DEV), enc["attention_mask"].to(DEV), torch.tensor([labels[k] for k in j]).to(DEV)


def evaluate(model, tok, frame, loss_fn):
    model.eval()
    probs, total = [], 0.0
    with torch.no_grad():
        for ids, mask, y in batches(tok, frame, False, None):
            logits = model(ids, mask).float()
            total += loss_fn(logits, y).item() * len(y)
            probs.append(torch.softmax(logits, -1)[:, 1].cpu().numpy())
    return np.concatenate(probs), total / len(frame)


def train_one(name, dataset):
    train, val, test = DATA[dataset]
    torch.manual_seed(SEED)
    rng = np.random.RandomState(SEED)
    tok = AutoTokenizer.from_pretrained(BERT)
    model = BUILD[name](tok).to(DEV)
    cfg = SETTINGS[name]
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    steps = EPOCHS * ((len(train) + BATCH - 1) // BATCH)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    loss_fn = nn.CrossEntropyLoss()
    train_loss, val_loss, best, best_state = [], [], float("inf"), None
    for epoch in range(EPOCHS):
        model.train()
        t0, running, seen = time.time(), 0.0, 0
        for ids, mask, y in batches(tok, train, True, rng):
            loss = loss_fn(model(ids, mask).float(), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            opt.zero_grad()
            running += loss.item() * len(y)
            seen += len(y)
        _, vloss = evaluate(model, tok, val, loss_fn)
        train_loss.append(running / seen)
        val_loss.append(vloss)
        print(f"  {dataset} {name} epoch {epoch + 1}: train loss {running / seen:.4f} | validation loss {vloss:.4f} | {time.time() - t0:.0f}s", flush=True)
        if vloss < best:
            best, best_state = vloss, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    probs, test_loss = evaluate(model, tok, test, loss_fn)
    return {"y": test["label"].values, "p": probs, "train_loss": train_loss, "val_loss": val_loss, "test_loss": test_loss}

## 3. Train all six runs (2 datasets x 3 models)

The per-epoch training and validation loss printed here are exactly the values plotted in the loss-curve figure below.

In [ ]:
RESULTS = {}
for dataset in ("isot", "liar"):
    for name in ("bilstm", "bert", "bert_bilstm"):
        print(f"== {dataset} / {name}")
        RESULTS[(dataset, name)] = train_one(name, dataset)

## 4. Test-set metrics

In [ ]:
rows = []
for (dataset, name), r in RESULTS.items():
    pred = (r["p"] >= 0.5).astype(int)
    rows.append({"dataset": "ISOT" if dataset == "isot" else "LIAR2", "model": name,
                 "accuracy": accuracy_score(r["y"], pred), "precision": precision_score(r["y"], pred, zero_division=0),
                 "recall": recall_score(r["y"], pred, zero_division=0), "f1": f1_score(r["y"], pred, zero_division=0),
                 "auc": roc_auc_score(r["y"], r["p"]), "test_loss": r["test_loss"]})
metrics = pd.DataFrame(rows).round(4)
metrics.to_csv(OUT / "metrics.csv", index=False)
metrics

## 5. Loss curves (Fig. 4.4)

Blue is the training loss and orange the validation loss after each epoch. The red star marks the epoch with the lowest validation loss, whose weights are used for testing.

**How to read it**
* **ISOT:** training loss falls steadily while validation loss stays low and almost flat, so the models fit the data well. After the best epoch the validation loss stops improving or rises slightly while the training loss keeps falling, which is the start of overfitting and the reason the best epoch is restored.
* **LIAR2:** training loss keeps falling but validation loss does **not**. For all three models it is lowest at epoch 1 and then rises, so the models start to memorise the training statements after one epoch instead of learning patterns that carry over. This matches the lower LIAR2 accuracy of about 0.66 to 0.69.

In [ ]:
DATASETS = [("isot", "ISOT"), ("liar", "LIAR2")]
MODELS = [("bilstm", "BiLSTM", "#d95f02"), ("bert", "BERT", "#1b9e77"), ("bert_bilstm", "BERT+BiLSTM", "#7570b3")]

fig, axes = plt.subplots(2, 3, figsize=(12, 7.5))
for r, (ds, ds_label) in enumerate(DATASETS):
    for c, (m, m_label, _) in enumerate(MODELS):
        d = RESULTS[(ds, m)]
        epochs = np.arange(1, EPOCHS + 1)
        best = int(np.argmin(d["val_loss"]))
        ax = axes[r][c]
        ax.plot(epochs, d["train_loss"], marker="o", color="#1f77b4", label="Training loss")
        ax.plot(epochs, d["val_loss"], marker="s", color="#ff7f0e", label="Validation loss")
        ax.scatter(epochs[best], d["val_loss"][best], marker="*", s=220, color="#d62728", zorder=5, label="Best validation epoch")
        for e, t, v in zip(epochs, d["train_loss"], d["val_loss"]):
            above = t >= v
            ax.annotate(f"{t:.3f}", (e, t), textcoords="offset points", xytext=(0, 9 if above else -14), ha="center", fontsize=7, color="#1f77b4")
            ax.annotate(f"{v:.3f}", (e, v), textcoords="offset points", xytext=(0, -14 if above else 9), ha="center", fontsize=7, color="#ff7f0e")
        ax.margins(y=0.2)
        ax.set_xticks(epochs)
        ax.set_title(f"{ds_label}: {m_label}", fontsize=11)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Cross-entropy loss")
        ax.grid(alpha=0.3)
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=10, frameon=False)
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig(OUT / "Loss.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
table = pd.DataFrame([{"dataset": ds_label, "model": m_label, "epoch": e + 1,
                       "train_loss": RESULTS[(ds, m)]["train_loss"][e], "val_loss": RESULTS[(ds, m)]["val_loss"][e]}
                      for ds, ds_label in DATASETS for m, m_label, _ in MODELS for e in range(EPOCHS)]).round(4)
table.to_csv(OUT / "loss_values.csv", index=False)
table

## 6. Confusion matrices, ROC and precision-recall curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 6.5))
for r, (ds, ds_label) in enumerate(DATASETS):
    for c, (m, m_label, _) in enumerate(MODELS):
        d = RESULTS[(ds, m)]
        cm = confusion_matrix(d["y"], (d["p"] >= 0.5).astype(int), labels=[0, 1])
        ax = axes[r][c]
        ax.imshow(cm, cmap="Blues")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
        ax.set_xticks([0, 1], ["Real", "Fake"])
        ax.set_yticks([0, 1], ["Real", "Fake"])
        ax.set_title(f"{ds_label}: {m_label}", fontsize=10)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
fig.tight_layout()
fig.savefig(OUT / "Confusion.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
for kind in ("roc", "pr"):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, (ds, ds_label) in zip(axes, DATASETS):
        for m, m_label, colour in MODELS:
            d = RESULTS[(ds, m)]
            if kind == "roc":
                x, y, _ = roc_curve(d["y"], d["p"])
                ax.plot(x, y, color=colour, label=f"{m_label} (AUC {auc(x, y):.3f})")
            else:
                prec, rec, _ = precision_recall_curve(d["y"], d["p"])
                ax.plot(rec, prec, color=colour, label=f"{m_label} (AUC {auc(rec, prec):.3f})")
        if kind == "roc":
            ax.plot([0, 1], [0, 1], "--", color="grey", linewidth=1)
            ax.set_xlabel("False positive rate")
            ax.set_ylabel("True positive rate")
        else:
            ax.set_xlabel("Recall")
            ax.set_ylabel("Precision")
        ax.set_title(ds_label)
        ax.legend(loc="lower right" if kind == "roc" else "lower left", fontsize=8)
        ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / ("ROC.png" if kind == "roc" else "PR_Curve.png"), dpi=200, bbox_inches="tight")
    plt.show()

## 7. Download everything

This packs the figures and the CSV files into one zip.

In [ ]:
import shutil
shutil.make_archive("fake_news_results", "zip", OUT)
try:
    from google.colab import files
    files.download("fake_news_results.zip")
except ImportError:
    print("Saved fake_news_results.zip")